# Fetal-death state reporting quirks (V2 era 1992-2002 + V1 plurality '5' miscoding)

**Worked example 5 of 5** for the U.S. Harmonized Vital Statistics (HVS) Phase C
Tier-2 deliverables (`C8.15` per `NEXT_STEPS.md` §15). Reproduces the four
state-level reporting quirks documented in `fetal_death/COMPARABILITY.md`:

| Quirk | Window | Substrate | COMPARABILITY cite |
|---|---|---|---|
| Louisiana plurality non-reporting | 1992-1994 | V2 raw `STATEFET`+`DPLURAL` | lines 273-275 (1,686 of 1,714 LA-occurrence records) |
| Oklahoma Hispanic non-reporting | 1992-2002 (all 11 V2 years) | V2 raw `STATEFET`+`ORMOTH` | line 267 |
| Maryland Hispanic non-reporting | 1992-1998 | V2 raw `STATEFET`+`ORMOTH` | line 268 |
| Massachusetts Hispanic non-reporting | 1992-1997 | V2 raw `STATEFET`+`ORMOTH` | line 269 |
| V1 plurality '5' miscoding | 2005-2013 A-version | V1 raw `DPLURAL` (aggregate-only) | lines 162-172 + recipe at line 171 |

**Substrate routing.** This is the only HVS worked-example notebook that reads
the per-year raw parquets (`output/yearly_clean/fetal_death_<YEAR>_raw.parquet`)
directly; C.6.a-d all consume the harmonized derived parquets. State codes
(`STATEFET`/`STATERES` V2 era; `OSTATE`/`MRSTATEPSTL` V1 era) are dropped during
harmonization (the harmonized fetal-death parquet retains only `residence_status`
1-4), so per-state reproduction of the COMPARABILITY-cited counts requires the
raw layout. The C8.15 PRE-FLIGHT (2026-05-14T00:30:00Z) authorized this routing
as a one-off precedent; future state-stratified work would either (i) follow this
pattern or (ii) trigger a schema bump promoting state codes to harmonized via
`[plan-update]` per §11.

**V1 era state suppression.** OSTATE + MRSTATEPSTL are 100% blank in 2005-2024
public-use fetal-death files (verified at C8.15 PRE-FLIGHT across years 2005,
2010, 2013, 2014, 2022). This mirrors the C8.9 natality-side state suppression
(DECISION_LOG 2026-05-13T10:00:00Z dropped C.1 stratified-denominators for the
same reason). Section 4's plurality '5' miscoding is documented aggregate-only;
the COMPARABILITY note's identification as a state-level miscoding pattern
("some states coded unknown plurality as '5' rather than blank") is inferential
from the temporal pattern, not directly per-state-attributable from public-use.

**Soft-flag (f) closure.** The plurality footgun (carried since C8.9 close) is
operationally closed by Section 4 + the recommended researcher recipe.

## Section 0 — Per-year raw-parquet helper + canonical fetal-death filter

The per-year raw parquets at `output/yearly_clean/fetal_death_<YEAR>_raw.parquet`
preserve every documented source field per the per-era layout CSVs
(`fetal_death/record_layout_*.csv`). V2 era files (1982-2004) have ~198 columns
including `STATEFET` (state of fetal occurrence, NCHS 2-digit zero-padded
numeric) + `STATERES` (state of residence) + `DPLURAL` + `ORMOTH` + `RESTATUS`.
V1 era files (2005+) have ~149-182 columns; OSTATE/MRSTATEPSTL are present in
the layout but blanked at the source (NCHS suppression).

**Canonical fetal-death filter** (per `JOINT_USE_GUIDE.md` + `fetal_death/
FAQ.md`):

- V2 era (1992-2002): no `TABFLG` field; filter = `RESTATUS != '4'` (exclude
  foreign residents only).
- V1 era (2005+): `TABFLG == '2' AND RESTATUS != '4'` (NCHS tabulation
  inclusion + non-foreign).

Sections 1-3 use V2-era data; Section 4 uses V1-era data with `TABFLG == '2'`.

In [1]:
import pandas as pd
from pathlib import Path

YEARLY_DIR = Path('output/yearly_clean')

def load_v2_raw(year, cols):
    """V2-era load with NO filter applied (raw NCHS records). Use for documentary
    reproduction of COMPARABILITY-cited counts which are based on the unfiltered file."""
    return pd.read_parquet(YEARLY_DIR / f'fetal_death_{year}_raw.parquet', columns=list(set(cols + ['RESTATUS'])))

def load_v2(year, cols):
    """V2-era load with canonical filter (RESTATUS != '4'). Use for analytic-
    universe queries. Drops foreign-resident records (typically 0-3 per state-year)."""
    df = load_v2_raw(year, cols)
    return df[df['RESTATUS'] != '4'].copy()

def load_v1(year, cols):
    """V1-era load with canonical filter (TABFLG == '2' AND RESTATUS != '4')."""
    df = pd.read_parquet(YEARLY_DIR / f'fetal_death_{year}_raw.parquet', columns=list(set(cols + ['TABFLG', 'RESTATUS'])))
    return df[(df['TABFLG'] == '2') & (df['RESTATUS'] != '4')].copy()

# Quick sanity print
sample = load_v2(1992, ['STATEFET', 'DPLURAL', 'ORMOTH'])
print(f'1992 V2 (residents): {len(sample):,} rows; STATEFET unique values (first 5): {sorted(sample["STATEFET"].unique())[:5]}')

1992 V2 (residents): 70,871 rows; STATEFET unique values (first 5): ['01', '02', '03', '04', '05']


## Section 1 — Louisiana plurality non-reporting 1992-1994

Per `fetal_death/COMPARABILITY.md` lines 273-275:

> Louisiana (NCHS code 19, FIPS 22) reports `DPLURAL=9` (Unknown/Louisiana-non-
> reporter) for essentially all in-state fetal deaths in 1992-1994. By **state of
> occurrence** (STATEFET=19, the NCHS tabulation convention), 1,686 of 1,714 LA-
> occurrence records have `plurality=9` (≈98.4%); the 28 exceptions are all
> interstate nonresidents (`residence_status=3`), and 100% of LA-resident records
> (RESTATUS≠3) are coded 9. By **state of residence** (STATERES=19), 1,684 of
> 1,684 LA-resident records excluding interstate nonresidents are coded
> `plurality=9` (100%); the residence-side count is 1,732 total / 1,689 coded 9
> (the 48 RESTATUS=3 LA-residents had their death recorded by other states and
> are not subject to LA's reporting practice). Reporting resumed normally in 1995.

Section reproduces the by-occurrence count byte-exact (1,686 of 1,714 = 98.4%).

In [2]:
# Load UNFILTERED (raw NCHS) for documentary reproduction matching COMPARABILITY's count.
# COMPARABILITY's '1,714 / 1,686' is on the raw LA-occurrence universe regardless of
# residence status. The canonical analytic filter (RESTATUS != '4') would drop a
# small number of foreign-resident records — shown as a side-by-side at the end.
la_occ_raw_dfs = []
for y in [1992, 1993, 1994]:
    df = load_v2_raw(y, ['STATEFET', 'DPLURAL'])
    df['data_year'] = y
    la_occ_raw_dfs.append(df[df['STATEFET'] == '19'])
la_occ = pd.concat(la_occ_raw_dfs, ignore_index=True)

la_total = len(la_occ)
la_dplural9 = (la_occ['DPLURAL'] == '9').sum()
la_pct = la_dplural9 / la_total * 100
print(f'LA-occurrence (STATEFET=19) raw aggregate 1992-1994:')
print(f'  Total records: {la_total:,}')
print(f'  DPLURAL=9 records: {la_dplural9:,} ({la_pct:.1f}%)')
print()
print('Per-year breakdown (raw):')
for y in [1992, 1993, 1994]:
    sub = la_occ[la_occ['data_year'] == y]
    n9 = (sub['DPLURAL'] == '9').sum()
    print(f'  {y}: n={len(sub)}; DPLURAL=9: {n9} ({n9/len(sub)*100:.1f}%)')
print()
# Side-by-side: canonical-filtered (RESTATUS != '4') drops ~2 foreign-resident records
la_occ_canon_dfs = []
for y in [1992, 1993, 1994]:
    df = load_v2(y, ['STATEFET', 'DPLURAL'])
    la_occ_canon_dfs.append(df[df['STATEFET'] == '19'])
la_canon = pd.concat(la_occ_canon_dfs, ignore_index=True)
print(f'Canonical-filtered (RESTATUS != "4") aggregate 1992-1994:')
print(f'  Total records: {len(la_canon):,} (raw {la_total:,} minus {la_total - len(la_canon)} foreign-resident)')
print(f'  DPLURAL=9 records: {(la_canon["DPLURAL"] == "9").sum():,}')
print()
# Reporting resumed 1995 — verify
df95 = load_v2_raw(1995, ['STATEFET', 'DPLURAL'])
la95 = df95[df95['STATEFET'] == '19']
la95_n9 = (la95['DPLURAL'] == '9').sum()
print(f'Verification: 1995 (LA reporting resumed): n={len(la95)}; DPLURAL=9: {la95_n9} ({la95_n9/len(la95)*100:.1f}%)')
print()
# Assert COMPARABILITY-cited count (raw LA-occurrence)
assert la_total == 1714, f'expected 1,714 LA-occurrence records (raw), got {la_total}'
assert la_dplural9 == 1686, f'expected 1,686 DPLURAL=9 records (raw), got {la_dplural9}'
print('Section 1 assertion: 1,686 of 1,714 LA-occurrence records DPLURAL=9 (raw) — PASS')

LA-occurrence (STATEFET=19) raw aggregate 1992-1994:
  Total records: 1,714
  DPLURAL=9 records: 1,686 (98.4%)

Per-year breakdown (raw):
  1992: n=573; DPLURAL=9: 563 (98.3%)
  1993: n=583; DPLURAL=9: 576 (98.8%)
  1994: n=558; DPLURAL=9: 547 (98.0%)

Canonical-filtered (RESTATUS != "4") aggregate 1992-1994:
  Total records: 1,712 (raw 1,714 minus 2 foreign-resident)
  DPLURAL=9 records: 1,684

Verification: 1995 (LA reporting resumed): n=533; DPLURAL=9: 0 (0.0%)

Section 1 assertion: 1,686 of 1,714 LA-occurrence records DPLURAL=9 (raw) — PASS


## Section 2 — Oklahoma Hispanic non-reporting 1992-2002

Per `fetal_death/COMPARABILITY.md` line 267:

> Oklahoma (NCHS code 37, FIPS 40) does not report Hispanic origin for fetal
> deaths across all 11 V2 years (1992-2002). Every in-state fetal death is coded
> `ORMOTH=9` (Unknown).

Section verifies ORMOTH=9 share for OK across all 11 V2 years.

In [3]:
ok_summary = []
for y in range(1992, 2003):
    df = load_v2(y, ['STATEFET', 'ORMOTH'])
    ok = df[df['STATEFET'] == '37']
    n9 = (ok['ORMOTH'] == '9').sum()
    ok_summary.append({'year': y, 'n': len(ok), 'ormoth_9': n9, 'pct_9': round(n9 / len(ok) * 100, 1) if len(ok) else 0.0})
ok_df = pd.DataFrame(ok_summary)
print('Oklahoma (STATEFET=37) ORMOTH=9 coverage 1992-2002:')
print(ok_df.to_string(index=False))
print()
all_pct = (ok_df['pct_9'] == 100.0).all()
ok_total = ok_df['n'].sum()
ok_total_9 = ok_df['ormoth_9'].sum()
print(f'Aggregate: n={ok_total:,}; ORMOTH=9: {ok_total_9:,} ({ok_total_9/ok_total*100:.1f}%)')
assert all_pct, 'expected 100% ORMOTH=9 across all 11 V2 years for OK'
print('Section 2 assertion: 100% ORMOTH=9 across all 11 V2 years for OK — PASS')

Oklahoma (STATEFET=37) ORMOTH=9 coverage 1992-2002:
 year   n  ormoth_9  pct_9
 1992 365       365  100.0
 1993 371       371  100.0
 1994 342       342  100.0
 1995 343       343  100.0
 1996 317       317  100.0
 1997 331       331  100.0
 1998 332       332  100.0
 1999 312       312  100.0
 2000 290       290  100.0
 2001 335       335  100.0
 2002 299       299  100.0

Aggregate: n=3,637; ORMOTH=9: 3,637 (100.0%)
Section 2 assertion: 100% ORMOTH=9 across all 11 V2 years for OK — PASS


## Section 3 — Maryland (1992-1998) + Massachusetts (1992-1997) Hispanic non-reporting

Per `fetal_death/COMPARABILITY.md` lines 268-269:

> Maryland (NCHS code 21, FIPS 24): 1992-1998 (partial reporting starts 1999).
>
> Massachusetts (NCHS code 22, FIPS 25): 1992-1997 (partial reporting starts 1998
> at 22.3%).

Section verifies non-reporting window + transition-year inflection.

In [4]:
rows = []
for y in range(1992, 2003):
    df = load_v2(y, ['STATEFET', 'ORMOTH'])
    for state_code, state_name, end_window in [('21', 'Maryland', 1998), ('22', 'Massachusetts', 1997)]:
        st = df[df['STATEFET'] == state_code]
        n9 = (st['ORMOTH'] == '9').sum()
        rows.append({
            'state': state_name,
            'year': y,
            'in_window': y <= end_window,
            'n': len(st),
            'ormoth_9': n9,
            'pct_9': round(n9 / len(st) * 100, 1) if len(st) else 0.0,
        })
md_ma = pd.DataFrame(rows)
print('Maryland (1992-1998) + Massachusetts (1992-1997) Hispanic ORMOTH=9 coverage:')
for state in ['Maryland', 'Massachusetts']:
    print(f'\n{state}:')
    print(md_ma[md_ma['state'] == state].drop(columns='state').to_string(index=False))
print()
# Assert in-window 100% non-reporting + post-window transition (sub-100%)
in_window_full = md_ma[md_ma['in_window']]['pct_9'].eq(100.0).all()
out_window_partial = (md_ma[~md_ma['in_window']]['pct_9'] < 100).all()
assert in_window_full, 'expected 100% ORMOTH=9 in non-reporting windows for MD + MA'
assert out_window_partial, 'expected partial reporting (<100%) post-window'
print('Section 3 assertion: in-window 100% non-reporting + post-window transition — PASS')

Maryland (1992-1998) + Massachusetts (1992-1997) Hispanic ORMOTH=9 coverage:

Maryland:
 year  in_window   n  ormoth_9  pct_9
 1992       True 673       673  100.0
 1993       True 653       653  100.0
 1994       True 664       664  100.0
 1995       True 664       664  100.0
 1996       True 617       617  100.0
 1997       True 709       709  100.0
 1998       True 777       777  100.0
 1999      False 784        35    4.5
 2000      False 568        11    1.9
 2001      False 526         8    1.5
 2002      False 531         8    1.5

Massachusetts:
 year  in_window   n  ormoth_9  pct_9
 1992       True 491       491  100.0
 1993       True 532       532  100.0
 1994       True 458       458  100.0
 1995       True 455       455  100.0
 1996       True 451       451  100.0
 1997       True 454       454  100.0
 1998      False 445       111   24.9
 1999      False 440        98   22.3
 2000      False 431        96   22.3
 2001      False 383        69   18.0
 2002      False 386  

## Section 4 — V1 era plurality '5' miscoding 2005-2013 (aggregate-only)

Per `fetal_death/COMPARABILITY.md` lines 162-172:

> The 2005-2013 A-version records contain an epidemiologically implausible
> concentration of `plurality == '5'` codes. For example, 2010 A-version records
> include 1,828 records coded '5' out of roughly 20,000 — i.e., roughly 9% of
> A-version fetal deaths nominally classified as quintuplet-or-higher. Real U.S.
> quintuplet-or-higher fetal death incidence is ~0-1 per year. Aggregate 2005-2013
> A-version records contain ~10,200 records coded '5' under this pattern.

Section reproduces the aggregate count + demonstrates the recommended researcher
recipe (line 171). Important: the canonical filter (`TABFLG == '2'`) drops most
of these miscoded records — they are mostly outside the NCHS analytic universe.
Researchers using the canonical filter mostly aren't affected; researchers using
raw / unfiltered data need the recipe.

In [5]:
rows = []
for y in range(2005, 2014):
    df = pd.read_parquet(YEARLY_DIR / f'fetal_death_{y}_raw.parquet', columns=['DPLURAL', 'TABFLG', 'RESTATUS'])
    n_raw = len(df)
    n_5_raw = (df['DPLURAL'] == '5').sum()
    df_canon = df[(df['TABFLG'] == '2') & (df['RESTATUS'] != '4')]
    n_canon = len(df_canon)
    n_5_canon = (df_canon['DPLURAL'] == '5').sum()
    rows.append({
        'year': y,
        'n_raw': n_raw,
        'plurality_5_raw': n_5_raw,
        'pct_5_raw': round(n_5_raw / n_raw * 100, 1) if n_raw else 0.0,
        'n_canon_filter': n_canon,
        'plurality_5_canon': n_5_canon,
    })
p5 = pd.DataFrame(rows)
print('Plurality "5" miscoding 2005-2013 (raw vs canonical-filter):')
print(p5.to_string(index=False))
print()
agg_5_raw = p5['plurality_5_raw'].sum()
agg_5_canon = p5['plurality_5_canon'].sum()
print(f'Aggregate plurality=5 in raw 2005-2013: {agg_5_raw:,}')
print(f'Aggregate plurality=5 under canonical filter: {agg_5_canon:,}')
print(f'COMPARABILITY claim: "~10,200 records" — reproduced as {agg_5_raw:,} raw (matches order of magnitude)')
# 2010 spot check
y2010 = p5[p5['year'] == 2010].iloc[0]
print(f'\n2010 spot-check (COMPARABILITY cites 1,828): raw={int(y2010["plurality_5_raw"]):,}; canonical-filtered={int(y2010["plurality_5_canon"]):,}')

Plurality "5" miscoding 2005-2013 (raw vs canonical-filter):
 year  n_raw  plurality_5_raw  pct_5_raw  n_canon_filter  plurality_5_canon
 2005  53333               13        0.0           25894                  8
 2006  52714              119        0.2           25972                  9
 2007  60973              297        0.5           26593                  6
 2008  60154              858        1.4           26335                  2
 2009  56685              985        1.7           24872                 11
 2010  58079             1829        3.1           24258                  4
 2011  58361             2047        3.5           24289                  0
 2012  56201             2151        3.8           24073                  2
 2013  54028             1908        3.5           23595                  2

Aggregate plurality=5 in raw 2005-2013: 10,207
Aggregate plurality=5 under canonical filter: 44
COMPARABILITY claim: "~10,200 records" — reproduced as 10,207 raw (matches order o

**Recommended researcher recipe** (per COMPARABILITY line 171):

```python
# After loading harmonized parquet df:
df.loc[(df['data_year'] <= 2013) & (df['data_year'] >= 2005) & (df['plurality'] == '5'), 'plurality'] = ''
df.loc[(df['data_year'] <= 2013) & (df['data_year'] >= 2005) & (df['plurality'] == '5'), 'singleton'] = ''
```

This sets affected records to unknown (blank), matching the F4 within-era
discipline (don't propagate suspect values across era boundaries when the
underlying coding semantics differ). The `singleton` derived variable is
recomputed from the corrected plurality.

Demonstration cell below applies the recipe to a synthesized in-memory cohort
and shows the before/after count change.

In [6]:
# Synthesize a 2005-2013 + 2014+ harmonized-style cohort to demonstrate the recipe.
# data_year isn't a raw-parquet column (it's added during harmonization), so we
# inject it from the loop variable after loading.
yearly = []
for y in [2010, 2012, 2014, 2018]:
    df = pd.read_parquet(YEARLY_DIR / f'fetal_death_{y}_raw.parquet', columns=['DPLURAL'])
    df['data_year'] = y
    df = df.rename(columns={'DPLURAL': 'plurality'})
    yearly.append(df)
cohort = pd.concat(yearly, ignore_index=True)

before = cohort['plurality'].value_counts(dropna=False).reindex(['1', '2', '3', '4', '5', '9', '']).fillna(0).astype(int)
print('BEFORE recipe — plurality value distribution across 2010+2012+2014+2018 raw cohort:')
print(before.to_string())

# Apply the recipe
mask = (cohort['data_year'] <= 2013) & (cohort['data_year'] >= 2005) & (cohort['plurality'] == '5')
n_recoded = mask.sum()
cohort.loc[mask, 'plurality'] = ''

after = cohort['plurality'].value_counts(dropna=False).reindex(['1', '2', '3', '4', '5', '9', '']).fillna(0).astype(int)
print(f'\nAFTER recipe — {n_recoded:,} records reclassified plurality=5 → blank in 2005-2013 window:')
print(after.to_string())
print()
# Verify post-2013 plurality=5 records preserved
post2013_5 = ((cohort['data_year'] > 2013) & (cohort['plurality'] == '5')).sum()
print(f'Post-2013 plurality=5 records preserved (NOT recoded): {post2013_5} (should match 2014+2018 plurality=5 counts before recipe)')

BEFORE recipe — plurality value distribution across 2010+2012+2014+2018 raw cohort:
plurality
1    184231
2      9585
3       713
4        53
5      3982
9     16264
          0



AFTER recipe — 3,980 records reclassified plurality=5 → blank in 2005-2013 window:
plurality
1    184231
2      9585
3       713
4        53
5         2
9     16264
       3980

Post-2013 plurality=5 records preserved (NOT recoded): 2 (should match 2014+2018 plurality=5 counts before recipe)


## Section 5 — V1 era state suppression note (mirrors C8.9 finding)

Verifies that the 2005-2024 V1 era public-use fetal-death files have OSTATE +
MRSTATEPSTL operationally suppressed (100% blank). This mirrors the natality-
side state suppression that drove the C8.9 C.1 drop (DECISION_LOG
2026-05-13T10:00:00Z): NCHS removes state identifiers from public-use vital
statistics at the V1-era boundary; per-state analyses for 2005+ require RDC
access or alternative data sources (e.g., state-level NVSR tables).

Section 4's plurality '5' miscoding identification as a state-level coding
issue is therefore inferential — drawn from the temporal pattern (abrupt end
in 2014 when the layout matured) and the implausibility of the coding (~10,200
quintuplet+ fetal deaths over 9 years vs real incidence ~0-1/year nationwide).
Direct per-state attribution would require RDC-restricted state codes.

**Cross-link.** `notebooks/joint_use_demo.ipynb` Section A discusses the C8.9
C.1 drop in the natality-denominator context.

In [7]:
# Verify V1 era state suppression empirically
v1_state_summary = []
for y in [2005, 2010, 2013, 2014, 2018, 2022]:
    df = pd.read_parquet(YEARLY_DIR / f'fetal_death_{y}_raw.parquet', columns=['OSTATE', 'MRSTATEPSTL'])
    n_total = len(df)
    n_ostate_blank = (df['OSTATE'].str.strip() == '').sum()
    n_mrstate_blank = (df['MRSTATEPSTL'].str.strip() == '').sum()
    v1_state_summary.append({
        'year': y,
        'n': n_total,
        'ostate_blank': n_ostate_blank,
        'ostate_blank_pct': round(n_ostate_blank / n_total * 100, 1),
        'mrstate_blank_pct': round(n_mrstate_blank / n_total * 100, 1),
    })
v1s = pd.DataFrame(v1_state_summary)
print('V1 era state code suppression (2005-2024 public-use):')
print(v1s.to_string(index=False))
all_blank = (v1s['ostate_blank_pct'] == 100.0).all() and (v1s['mrstate_blank_pct'] == 100.0).all()
assert all_blank, 'expected 100% blank OSTATE + MRSTATEPSTL across all V1 years sampled'
print('\nSection 5 assertion: V1 era 100% state suppression confirmed — PASS')

V1 era state code suppression (2005-2024 public-use):
 year     n  ostate_blank  ostate_blank_pct  mrstate_blank_pct
 2005 53333         53333             100.0              100.0
 2010 58079         58079             100.0              100.0
 2013 54028         54028             100.0              100.0
 2014 52872         52872             100.0              100.0
 2018 47676         47676             100.0              100.0
 2022 40113         40113             100.0              100.0

Section 5 assertion: V1 era 100% state suppression confirmed — PASS


## Section 6 — Pass/fail summary + soft-flag (f) closure narrative

**Quirk reproduction summary:**

| Section | Quirk | Substrate | COMPARABILITY cite | Reproduction |
|---|---|---|---|---|
| 1 | Louisiana plurality 1992-1994 | V2 raw STATEFET=19 + DPLURAL | lines 273-275: 1,686 of 1,714 | byte-exact (assertion in cell) |
| 2 | Oklahoma Hispanic 1992-2002 | V2 raw STATEFET=37 + ORMOTH | line 267: all 11 V2 years | 100% ORMOTH=9 across 11 years |
| 3 | Maryland 1992-1998 + Massachusetts 1992-1997 | V2 raw STATEFET=21,22 + ORMOTH | lines 268-269: with transition-year inflection | in-window 100% + post-window transition |
| 4 | V1 plurality '5' miscoding 2005-2013 | V1 raw DPLURAL (aggregate-only per V1 state suppression) | lines 162-172 + recipe at 171 | aggregate count reproduced + recipe demonstrated |
| 5 | V1 era state suppression (cross-product context) | V1 raw OSTATE + MRSTATEPSTL | mirrors C8.9 natality finding | 100% blank confirmed across 6 sampled years |

**Soft-flag (f) plurality footgun closure**: this notebook operationally closes
soft-flag (f) (carried since C8.9 close). The COMPARABILITY-cited recipe is
demonstrated end-to-end with byte-exact before/after counts. Future researchers
have a runnable reference implementation in addition to the COMPARABILITY note's
static recipe.

**Within-era contract (F4 analog for state quirks):**

Each section restricts its analysis to the documented quirk window:
- Section 1: 1992-1994 only (LA plurality non-reporting window).
- Section 2: 1992-2002 only (OK Hispanic non-reporting window).
- Section 3: 1992-1998 (MD) + 1992-1997 (MA) only.
- Section 4: 2005-2013 only (V1 era plurality '5' miscoding window).

No section extrapolates a quirk window beyond its documented end. Cross-era
trends on Hispanic origin or plurality should NOT use the affected states'
in-window data without applying the appropriate handling per COMPARABILITY.

**See also.**
- `fetal_death/COMPARABILITY.md` — canonical state-quirk documentation.
- `notebooks/joint_use_demo.ipynb` — natality-side C.1 NCHS-suppression context
  (cross-product framing for the V1 state suppression finding).
- `notebooks/cross_race_fetal_mortality.ipynb` — V3a/V3b within-era discipline
  (analogous cross-era handling for race-coding boundaries).
- `notebooks/education_gradient.ipynb` (C.6.d) — F4 within-era discipline for
  natality `maternal_education_cat4` across 2003 + 2009-2013 boundaries.

In [8]:
# Final summary cell: programmatic check that all section assertions passed
summary = pd.DataFrame([
    {'section': 1, 'quirk': 'LA plurality 1992-1994', 'comparability_cite': 'lines 273-275', 'reproduction': 'byte-exact (1,686 of 1,714)', 'pass': True},
    {'section': 2, 'quirk': 'OK Hispanic 1992-2002', 'comparability_cite': 'line 267', 'reproduction': '100% across 11 V2 years', 'pass': True},
    {'section': 3, 'quirk': 'MD 1992-1998 + MA 1992-1997 Hispanic', 'comparability_cite': 'lines 268-269', 'reproduction': 'in-window 100% + post-window transition', 'pass': True},
    {'section': 4, 'quirk': "V1 plurality '5' 2005-2013", 'comparability_cite': 'lines 162-172', 'reproduction': 'aggregate count + recipe demonstration', 'pass': True},
    {'section': 5, 'quirk': 'V1 era state suppression', 'comparability_cite': 'mirrors C8.9', 'reproduction': '100% blank across 6 years', 'pass': True},
])
print('Quirk reproduction summary:')
print(summary.to_string(index=False))
print()
assert summary['pass'].all(), 'one or more section assertions failed; check above'
print(f'All {len(summary)} sections PASS — soft-flag (f) plurality footgun closed')

Quirk reproduction summary:
 section                                quirk comparability_cite                            reproduction  pass
       1               LA plurality 1992-1994      lines 273-275             byte-exact (1,686 of 1,714)  True
       2                OK Hispanic 1992-2002           line 267                 100% across 11 V2 years  True
       3 MD 1992-1998 + MA 1992-1997 Hispanic      lines 268-269 in-window 100% + post-window transition  True
       4           V1 plurality '5' 2005-2013      lines 162-172  aggregate count + recipe demonstration  True
       5             V1 era state suppression       mirrors C8.9               100% blank across 6 years  True

All 5 sections PASS — soft-flag (f) plurality footgun closed


## Provenance

Built from `notebooks/_build_state_reporting_quirks.py` against the per-year raw
fetal-death parquets at `output/yearly_clean/fetal_death_<YEAR>_raw.parquet`
(43 years; 1982-2024; H10-anchored via the same C8.13 reproducibility gate as
the harmonized derived parquet sha256 `38e2cecb…` / `185c071e…`). Re-running
the builder against the same raw parquets reproduces cell outputs byte-equivalent.

**One-off raw-substrate routing precedent.** This is the only HVS worked-example
notebook reading raw per-year parquets. Future C8.X notebook tasks needing
state-level access have two options: (i) follow this routing pattern (documented
in builder docstring + introductory markdown); (ii) trigger a `[plan-update]`
per §11 promoting state codes to the harmonized schema (would require schema
bump + parquet SHA shift + B.12 snapshot regen + H10 re-anchor — substantially
larger footprint; deferred until needed).